# Banking Customer 360 Analytics
## 02 - Data Cleaning

Purpose: Clean and validate the raw banking tables before integrating them into the Customer 360 dataset.

## 1. Load Raw Tables

Load the raw banking tables so that cleaning and validation can be performed consistently before integration.


In [1]:
import pandas as pd
from pathlib import Path

file_path = Path("../data/raw/banking_customer360_synthetic.xlsx")

sheets = pd.read_excel(
    file_path,
    sheet_name=None
)

customers = sheets["Customers"].copy()
accounts = sheets["Accounts"].copy()
transactions = sheets["Transactions"].copy()
loans = sheets["Loans"].copy()
customer_service = sheets["Customer_Service"].copy()

print("Customers:", customers.shape)
print("Accounts:", accounts.shape)
print("Transactions:", transactions.shape)
print("Loans:", loans.shape)
print("Customer Service:", customer_service.shape)

Customers: (5000, 26)
Accounts: (7500, 6)
Transactions: (100000, 7)
Loans: (2500, 7)
Customer Service: (10000, 6)


## 2. Standardize Column Names

Standardize column names across all tables to make joins, analysis, and reusable code more reliable.


In [2]:
# Standardize column names
dataframes = {
    "Customers": customers,
    "Accounts": accounts,
    "Transactions": transactions,
    "Loans": loans,
    "Customer_Service": customer_service
}

for name, df in dataframes.items():
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )

    print(f"{name}:")
    print(df.columns.tolist())
    print()

Customers:
['customer_id', 'age', 'gender', 'country', 'occupation', 'annual_income', 'credit_score', 'tenure_years', 'join_date', 'total_transactions', 'total_transaction_value', 'avg_transaction_value', 'last_transaction_date', 'loan_count', 'total_loan_amount', 'late_loan_count', 'default_loan_count', 'complaint_count', 'escalated_count', 'account_count', 'total_balance', 'active_account_count', 'recency_days', 'products_number', 'churn', 'loan_to_income_ratio']

Accounts:
['account_id', 'customer_id', 'account_type', 'opening_date', 'status', 'balance']

Transactions:
['transaction_id', 'customer_id', 'account_id', 'transaction_date', 'transaction_type', 'amount', 'channel']

Loans:
['loan_id', 'customer_id', 'loan_type', 'loan_amount', 'interest_rate', 'start_date', 'repayment_status']

Customer_Service:
['interaction_id', 'customer_id', 'interaction_date', 'complaint_type', 'resolution_status', 'resolution_days']



## 3. Validate Data Types

Review the data types of each field and identify columns that need conversion before analysis.


In [3]:
for name, df in dataframes.items():
    print(f"\n{'=' * 60}")
    print(name)
    print(f"{'=' * 60}")
    print(df.dtypes)


Customers
customer_id                           str
age                                 int64
gender                                str
country                               str
occupation                            str
annual_income                       int64
credit_score                        int64
tenure_years                        int64
join_date                  datetime64[us]
total_transactions                  int64
total_transaction_value           float64
avg_transaction_value             float64
last_transaction_date      datetime64[us]
loan_count                          int64
total_loan_amount                 float64
late_loan_count                     int64
default_loan_count                  int64
complaint_count                     int64
escalated_count                     int64
account_count                       int64
total_balance                     float64
active_account_count                int64
recency_days                        int64
products_number        

## 4. Convert Date Columns

Convert date-related fields to datetime format so they can be used reliably for time-based analysis and feature engineering.


In [4]:
# Identify columns containing "date"

for name, df in dataframes.items():
    date_columns = [
        col for col in df.columns
        if "date" in col.lower()
    ]

    print(f"{name}: {date_columns}")

Customers: ['join_date', 'last_transaction_date']
Accounts: ['opening_date']
Transactions: ['transaction_date']
Loans: ['start_date']
Customer_Service: ['interaction_date']


In [5]:
# Convert identified date columns to datetime

for name, df in dataframes.items():
    date_columns = [
        col for col in df.columns
        if "date" in col.lower()
    ]

    for col in date_columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

print("Date conversion completed.")

Date conversion completed.


In [6]:
for name, df in dataframes.items():
    date_columns = [
        col for col in df.columns
        if "date" in col.lower()
    ]

    for col in date_columns:
        print(f"{name} → {col}: {df[col].dtype}")

Customers → join_date: datetime64[us]
Customers → last_transaction_date: datetime64[us]
Accounts → opening_date: datetime64[us]
Transactions → transaction_date: datetime64[us]
Loans → start_date: datetime64[us]
Customer_Service → interaction_date: datetime64[us]


## 5. Handle Missing Values

Identify missing values in the cleaned tables and determine which fields require treatment before analysis.


In [7]:
missing_summary = {}

for name, df in dataframes.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]

    missing_summary[name] = missing

    print(f"\n{'=' * 60}")
    print(name)
    print(f"{'=' * 60}")

    if missing.empty:
        print("No missing values.")
    else:
        print(missing)


Customers
No missing values.

Accounts
No missing values.

Transactions
account_id    22593
dtype: int64

Loans
No missing values.

Customer_Service
No missing values.


## 6. Validate Key Fields

Validate the primary and relationship keys to ensure records can be safely connected across the banking tables.


In [8]:
key_columns = {
    "Customers": ["customer_id"],
    "Accounts": ["account_id", "customer_id"],
    "Transactions": ["transaction_id", "customer_id", "account_id"],
    "Loans": ["loan_id", "customer_id"],
    "Customer_Service": ["interaction_id", "customer_id"]
}

for name, columns in key_columns.items():
    df = dataframes[name]

    print(f"\n{'=' * 60}")
    print(name)
    print(f"{'=' * 60}")

    for col in columns:
        print(
            f"{col}: "
            f"missing={df[col].isna().sum()}, "
            f"duplicates={df[col].duplicated().sum()}"
        )


Customers
customer_id: missing=0, duplicates=0

Accounts
account_id: missing=0, duplicates=0
customer_id: missing=0, duplicates=3622

Transactions
transaction_id: missing=0, duplicates=0
customer_id: missing=0, duplicates=95000
account_id: missing=22593, duplicates=92518

Loans
loan_id: missing=0, duplicates=0
customer_id: missing=0, duplicates=538

Customer_Service
interaction_id: missing=0, duplicates=0
customer_id: missing=0, duplicates=5697


## 7. Remove Duplicate Records

Remove exact duplicate records from the banking tables while preserving legitimate repeated customer and transaction activity.


In [9]:
duplicate_summary = {}

for name, df in dataframes.items():
    before = len(df)

    df.drop_duplicates(inplace=True)

    after = len(df)

    duplicate_summary[name] = before - after

    print(
        f"{name}: removed {before - after} duplicate rows "
        f"({before} → {after})"
    )

Customers: removed 0 duplicate rows (5000 → 5000)
Accounts: removed 0 duplicate rows (7500 → 7500)
Transactions: removed 0 duplicate rows (100000 → 100000)
Loans: removed 0 duplicate rows (2500 → 2500)
Customer_Service: removed 0 duplicate rows (10000 → 10000)


## 8. Final Cleaning Validation

Perform a final validation of row counts, missing values, duplicate rows, and data types before exporting the cleaned tables.


In [10]:
for name, df in dataframes.items():
    print(f"\n{'=' * 60}")
    print(name)
    print(f"{'=' * 60}")

    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Duplicate rows:", df.duplicated().sum())
    print("Missing values:", df.isna().sum().sum())


Customers
Rows: 5000
Columns: 26
Duplicate rows: 0
Missing values: 0

Accounts
Rows: 7500
Columns: 6
Duplicate rows: 0
Missing values: 0

Transactions
Rows: 100000
Columns: 7
Duplicate rows: 0
Missing values: 22593

Loans
Rows: 2500
Columns: 7
Duplicate rows: 0
Missing values: 0

Customer_Service
Rows: 10000
Columns: 6
Duplicate rows: 0
Missing values: 0


## 9. Save Cleaned Tables

Export the cleaned and validated banking tables to the processed-data folder for use in integration and feature engineering.


In [11]:
processed_path = Path("../data/processed")
processed_path.mkdir(parents=True, exist_ok=True)

for name, df in dataframes.items():
    file_name = name.lower().replace(" ", "_") + ".csv"
    output_file = processed_path / file_name

    df.to_csv(output_file, index=False)

    print(f"Saved: {output_file}")

Saved: ..\data\processed\customers.csv
Saved: ..\data\processed\accounts.csv
Saved: ..\data\processed\transactions.csv
Saved: ..\data\processed\loans.csv
Saved: ..\data\processed\customer_service.csv


In [12]:
for file in processed_path.iterdir():
    print(file.name)

accounts.csv
customers.csv
customer_service.csv
loans.csv
transactions.csv
